# Doppler-Based Positioning with LEO Satellites

This notebook estimates a stationary receiver's position from the Doppler curve of one Low Earth Orbit (LEO) satellite. It uses an ISS TLE snapshot fetched from CelesTrak, a Barcelona receiver, deterministic additive white Gaussian noise (AWGN), and an iterative linearized least-squares solution.

The TLE is embedded below so the experiment is repeatable and does not need network access. To refresh it, use the optional CelesTrak request in the setup cell and replace both pinned element lines. TLE accuracy degrades as the elements age, so refresh the snapshot for a new experiment date.

The measured carrier Doppler model is

$$
f_D = -\frac{f_c}{c}\left(\dot{\rho} + b_u\right) + \epsilon_f,
$$

where $\dot{\rho}$ is satellite-to-receiver range rate, $b_u = c\,\dot{\delta t}_u$ is the receiver clock drift expressed as an equivalent range rate in m/s, and $\epsilon_f$ is frequency measurement noise. The receiver is stationary, and the simulated truth has zero clock drift. The estimate solves for three Earth-fixed position coordinates and the clock term together.

The experiment uses a 10-degree elevation mask and a fixed random seed. Position corrections are linearized numerically around each current estimate; the clock-drift derivative is analytic. Skyfield handles TLE propagation and the satellite/receiver coordinate-frame transformation.

## LEO orbit

The LEO orbit in this notebook is the International Space Station (ISS) with NORAD catalog number 25544. The TLE snapshot is:

In [1]:
# Snapshot fetched from CelesTrak on 2026-10-05; keep fixed for reproducibility.
TLE_NAME = "ISS (ZARYA)"
TLE_LINE_1 = "1 25544U 98067A   26277.91750246  .00004575  00000+0  91962-4 0  9999"
TLE_LINE_2 = "2 25544  51.6317 117.0514 0006859 223.4758 136.5691 15.48736724588765"

Other TLE lines can be fetched from [CelesTrak](https://celestrak.org) with the following request:

```python
import requests
response = requests.get(
    "https://celestrak.org/NORAD/elements/gp.php",
    params={"CATNR": 25544, "FORMAT": "tle"}, timeout=20,
)
response.raise_for_status()
print(response.text)
``` 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from skyfield.api import EarthSatellite, load, wgs84

ts = load.timescale(builtin=True)
satellite = EarthSatellite(TLE_LINE_1, TLE_LINE_2, TLE_NAME, ts)
receiver = wgs84.latlon(41.3851, 2.1734, elevation_m=20)  # Barcelona

# Find the longest pass above the 10-degree mask during the TLE's first day of use.
search_minutes = np.arange(0, 24 * 60, 0.5)
search_times = ts.utc(2026, 10, 5, 0, search_minutes)
topocentric = (satellite - receiver).at(search_times)
elevation, _, _ = topocentric.altaz()
visible_indices = np.flatnonzero(elevation.degrees >= 10.0)
if visible_indices.size == 0:
    raise RuntimeError("No pass above the 10-degree elevation mask was found.")

passes = np.split(visible_indices, np.flatnonzero(np.diff(visible_indices) > 1) + 1)
pass_indices = max(passes, key=len)
pass_start = search_times[pass_indices[0]]
pass_end = search_times[pass_indices[-1]]
print(f"Selected {TLE_NAME} pass over Barcelona")
print(f"UTC: {pass_start.utc_datetime():%Y-%m-%d %H:%M:%S} to {pass_end.utc_datetime():%H:%M:%S}")
print(f"Visible duration: {(pass_end - pass_start) * 24 * 60:.1f} min")
print(f"Peak elevation: {elevation.degrees[pass_indices].max():.1f} deg")

Selected ISS (ZARYA) pass over Barcelona
UTC: 2026-10-05 09:02:30 to 09:08:30
Visible duration: 6.0 min
Peak elevation: 52.5 deg


In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pass_duration_s = (pass_end - pass_start) * 86_400.0
ground_track_seconds = np.linspace(0.0, pass_duration_s, max(2, int(np.ceil(pass_duration_s / 10.0)) + 1))
ground_track_times = ts.tt_jd(pass_start.tt + ground_track_seconds / 86_400.0)
ground_track_state = satellite.at(ground_track_times)
ground_subpoints = wgs84.subpoint(ground_track_state)
ground_track_latitude_deg = ground_subpoints.latitude.degrees
ground_track_longitude_deg = ground_subpoints.longitude.degrees

# Break the line at the antimeridian instead of drawing across the whole map.
plot_longitude_deg = ground_track_longitude_deg.copy()
plot_latitude_deg = ground_track_latitude_deg.copy()
antimeridian_crossings = np.flatnonzero(np.abs(np.diff(plot_longitude_deg)) > 180.0)
plot_longitude_deg[antimeridian_crossings + 1] = np.nan
plot_latitude_deg[antimeridian_crossings + 1] = np.nan

receiver_latitude_deg = receiver.latitude.degrees
receiver_longitude_deg = receiver.longitude.degrees
figure = go.Figure()
figure.add_trace(
    go.Scattergeo(
        lon=plot_longitude_deg,
        lat=plot_latitude_deg,
        mode="lines",
        name="ISS ground track",
        line={"color": "#137c8b", "width": 3},
    )
)
figure.add_trace(
    go.Scattergeo(
        lon=[ground_track_longitude_deg[0], ground_track_longitude_deg[-1]],
        lat=[ground_track_latitude_deg[0], ground_track_latitude_deg[-1]],
        mode="markers+text",
        name="Pass start / end",
        text=["Start", "End"],
        textposition="top center",
        marker={"size": 9, "color": "#e07a32", "symbol": ["circle", "x"]},
    )
)
figure.add_trace(
    go.Scattergeo(
        lon=[receiver_longitude_deg],
        lat=[receiver_latitude_deg],
        mode="markers+text",
        name="Receiver (Barcelona)",
        text=["Barcelona"],
        textposition="bottom right",
        marker={"size": 12, "color": "#bd3b32", "symbol": "star", "line": {"color": "white", "width": 1}},
    )
)
figure.update_geos(
    fitbounds="locations",
    projection_type="natural earth",
    showland=True,
    landcolor="#e7e5dc",
    showocean=True,
    oceancolor="#dcecf0",
    showcoastlines=True,
    coastlinecolor="#68777a",
    showcountries=True,
    countrycolor="#a4aaa5",
    lataxis_showgrid=True,
    lonaxis_showgrid=True,
)
figure.update_layout(
    title=f"{TLE_NAME} ground track during the visible pass",
    height=600,
    margin={"l": 10, "r": 10, "t": 55, "b": 10},
    legend={"orientation": "h", "yanchor": "bottom", "y": 0.01, "xanchor": "left", "x": 0.01},
)
figure.show()

print(f"Ground-track samples: {len(ground_track_longitude_deg)}")
print(f"Ground-track latitude span: {ground_track_latitude_deg.min():.1f} to {ground_track_latitude_deg.max():.1f} deg")
print(f"Ground-track longitude span: {ground_track_longitude_deg.min():.1f} to {ground_track_longitude_deg.max():.1f} deg")

Ground-track samples: 37
Ground-track latitude span: 31.9 to 45.6 deg
Ground-track longitude span: -5.6 to 17.1 deg


## 1. Simulate Doppler observations

Use the selected visible pass at 10-second spacing. The synthetic measurements combine the modeled carrier Doppler with zero-mean Gaussian frequency noise. The receiver clock drift is zero in the simulation, but remains an unknown in the estimator.

In [4]:
C = 299_792_458.0
CARRIER_HZ = 1_575_420_000.0  # GPS L1 frequency, used as a representative carrier
NOISE_STD_HZ = 0.25
RNG = np.random.default_rng(20261005)

WGS84_A_M = 6_378_137.0
WGS84_F = 1.0 / 298.257223563
WGS84_E2 = WGS84_F * (2.0 - WGS84_F)


def geodetic_to_ecef(latitude_deg, longitude_deg, height_m):
    latitude = np.deg2rad(latitude_deg)
    longitude = np.deg2rad(longitude_deg)
    prime_vertical_radius = WGS84_A_M / np.sqrt(1.0 - WGS84_E2 * np.sin(latitude) ** 2)
    x = (prime_vertical_radius + height_m) * np.cos(latitude) * np.cos(longitude)
    y = (prime_vertical_radius + height_m) * np.cos(latitude) * np.sin(longitude)
    z = (prime_vertical_radius * (1.0 - WGS84_E2) + height_m) * np.sin(latitude)
    return np.array([x, y, z])


def ecef_to_geodetic(position_m):
    x, y, z = np.asarray(position_m, dtype=float)
    longitude = np.arctan2(y, x)
    horizontal_radius = np.hypot(x, y)
    latitude = np.arctan2(z, horizontal_radius * (1.0 - WGS84_E2))
    for _ in range(8):
        prime_vertical_radius = WGS84_A_M / np.sqrt(1.0 - WGS84_E2 * np.sin(latitude) ** 2)
        height = horizontal_radius / np.cos(latitude) - prime_vertical_radius
        latitude = np.arctan2(
            z,
            horizontal_radius * (1.0 - WGS84_E2 * prime_vertical_radius / (prime_vertical_radius + height)),
        )
    prime_vertical_radius = WGS84_A_M / np.sqrt(1.0 - WGS84_E2 * np.sin(latitude) ** 2)
    height = horizontal_radius / np.cos(latitude) - prime_vertical_radius
    return np.rad2deg(latitude), np.rad2deg(longitude), height


def doppler_model(position_ecef_m, clock_bias_m_s, satellite_position_km, satellite_velocity_km_s, times):
    latitude_deg, longitude_deg, height_m = ecef_to_geodetic(position_ecef_m)
    site = wgs84.latlon(latitude_deg, longitude_deg, elevation_m=height_m)
    site_state = site.at(times)
    relative_position_m = (satellite_position_km - site_state.position.km) * 1_000.0
    relative_velocity_m_s = (satellite_velocity_km_s - site_state.velocity.km_per_s) * 1_000.0
    ranges_m = np.linalg.norm(relative_position_m, axis=0)
    range_rate_m_s = np.einsum("ij,ij->j", relative_position_m, relative_velocity_m_s) / ranges_m
    return -(CARRIER_HZ / C) * (range_rate_m_s + clock_bias_m_s)


# Sample the interior of the visible pass to avoid measurements exactly at the mask crossing.
pass_duration_s = (pass_end - pass_start) * 86_400.0
sample_seconds = np.arange(0.0, pass_duration_s + 1e-6, 10.0)
observation_times = ts.tt_jd(pass_start.tt + sample_seconds / 86_400.0)
satellite_state = satellite.at(observation_times)
satellite_position_km = satellite_state.position.km
satellite_velocity_km_s = satellite_state.velocity.km_per_s
true_position_ecef_m = geodetic_to_ecef(41.3851, 2.1734, 20.0)
true_clock_bias_m_s = 0.0

observed_elevation_deg = (satellite - receiver).at(observation_times).altaz()[0].degrees
assert np.all(observed_elevation_deg >= 10.0)
true_doppler_hz = doppler_model(
    true_position_ecef_m, true_clock_bias_m_s,
    satellite_position_km, satellite_velocity_km_s, observation_times,
)
observed_doppler_hz = true_doppler_hz + RNG.normal(0.0, NOISE_STD_HZ, size=true_doppler_hz.shape)
print(f"Measurements: {observed_doppler_hz.size} at 10-second spacing")
print(f"Elevation range: {observed_elevation_deg.min():.1f} to {observed_elevation_deg.max():.1f} deg")
print(f"Doppler range: {observed_doppler_hz.min():.1f} to {observed_doppler_hz.max():.1f} Hz")



Measurements: 37 at 10-second spacing
Elevation range: 11.9 to 52.5 deg
Doppler range: -34507.4 to 34384.4 Hz


In [5]:

elapsed_minutes = sample_seconds / 60.0
figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=("ISS carrier Doppler over the selected pass", "True vs. noisy Doppler measurements"),
)
figure.add_trace(
    go.Scatter(
        x=elapsed_minutes,
        y=true_doppler_hz,
        mode="lines",
        name="Noise-free truth",
        line={"color": "#137c8b", "width": 2},
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=elapsed_minutes,
        y=observed_doppler_hz,
        mode="markers",
        name="Noisy observations",
        marker={"color": "#e07a32", "size": 7, "opacity": 0.75},
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=elapsed_minutes,
        y=observed_doppler_hz - true_doppler_hz,
        mode="markers",
        name="True vs. Noisy observations",
        marker={"color": "#e07a32", "size": 7, "opacity": 0.75},
    ),
    row=2,
    col=1,
)

figure.update_yaxes(title_text="Doppler shift (Hz)", row=1, col=1)
figure.update_yaxes(title_text="Doppler shift error (Hz)", row=2, col=1)
figure.update_layout(
    title="Doppler values",
    height=600,
    template="plotly_white",
    hovermode="x unified",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0.0},
    margin={"l": 70, "r": 25, "t": 110, "b": 55},
)
figure.show()

## 2. Solve the linearized positioning problem

At each iteration, predict the Doppler at the current receiver state and solve the observed-minus-computed linear system. The three position columns use centered finite differences in Earth-fixed meters; the clock-rate column is $-f_c/c$. Updates are applied until the position correction is small.

In [6]:
initial_position_ecef_m = true_position_ecef_m + np.array([1_200.0, -900.0, 700.0])
initial_clock_bias_m_s = 4.0
estimate_position_ecef_m = initial_position_ecef_m.copy()
estimate_clock_bias_m_s = initial_clock_bias_m_s
position_difference_step_m = 1.0
iteration_history = []

for iteration in range(15):
    predicted_doppler_hz = doppler_model(
        estimate_position_ecef_m, estimate_clock_bias_m_s,
        satellite_position_km, satellite_velocity_km_s, observation_times,
    )
    residual_hz = observed_doppler_hz - predicted_doppler_hz
    jacobian = np.empty((residual_hz.size, 4))

    for coordinate in range(3):
        offset_m = np.zeros(3)
        offset_m[coordinate] = position_difference_step_m
        doppler_plus = doppler_model(
            estimate_position_ecef_m + offset_m, estimate_clock_bias_m_s,
            satellite_position_km, satellite_velocity_km_s, observation_times,
        )
        doppler_minus = doppler_model(
            estimate_position_ecef_m - offset_m, estimate_clock_bias_m_s,
            satellite_position_km, satellite_velocity_km_s, observation_times,
        )
        jacobian[:, coordinate] = (doppler_plus - doppler_minus) / (2.0 * position_difference_step_m)

    jacobian[:, 3] = -(CARRIER_HZ / C)
    update, _, rank, singular_values = np.linalg.lstsq(jacobian, residual_hz, rcond=None)
    estimate_position_ecef_m += update[:3]
    estimate_clock_bias_m_s += update[3]
    iteration_history.append((iteration + 1, np.linalg.norm(update[:3]), update[3]))
    if np.linalg.norm(update[:3]) < 1e-3 and abs(update[3]) < 1e-6:
        break

estimated_doppler_hz = doppler_model(
    estimate_position_ecef_m, estimate_clock_bias_m_s,
    satellite_position_km, satellite_velocity_km_s, observation_times,
)
final_residual_hz = observed_doppler_hz - estimated_doppler_hz
position_error_m = np.linalg.norm(estimate_position_ecef_m - true_position_ecef_m)

print(f"Iterations: {len(iteration_history)}")
print(f"Jacobian rank: {rank} / 4")
print(f"Position error: {position_error_m:.1f} m")
print(f"Estimated clock drift: {estimate_clock_bias_m_s:.3f} m/s (truth: {true_clock_bias_m_s:.1f} m/s)")
print(f"Residual RMS: {np.sqrt(np.mean(final_residual_hz**2)):.3f} Hz")
print(f"Jacobian condition number: {singular_values[0] / singular_values[-1]:.2e}")

assert rank == 4, "The observation geometry does not determine all four state parameters."

Iterations: 3
Jacobian rank: 4 / 4
Position error: 19.3 m
Estimated clock drift: 0.002 m/s (truth: 0.0 m/s)
Residual RMS: 0.208 Hz
Jacobian condition number: 8.83e+03


In [7]:
import pandas as pd
from IPython.display import Markdown, display

prefit_doppler_hz = doppler_model(
    initial_position_ecef_m,
    initial_clock_bias_m_s,
    satellite_position_km,
    satellite_velocity_km_s,
    observation_times,
)
prefit_residuals_hz = (observed_doppler_hz - prefit_doppler_hz).reshape(-1, 1)
observation_count = observed_doppler_hz.size
prefit_jacobian = np.empty((observation_count, 4))

for coordinate in range(3):
    offset_m = np.zeros(3)
    offset_m[coordinate] = position_difference_step_m
    doppler_plus = doppler_model(
        initial_position_ecef_m + offset_m,
        initial_clock_bias_m_s,
        satellite_position_km,
        satellite_velocity_km_s,
        observation_times,
    )
    doppler_minus = doppler_model(
        initial_position_ecef_m - offset_m,
        initial_clock_bias_m_s,
        satellite_position_km,
        satellite_velocity_km_s,
        observation_times,
    )
    prefit_jacobian[:, coordinate] = (doppler_plus - doppler_minus) / (2.0 * position_difference_step_m)

prefit_jacobian[:, 3] = -(CARRIER_HZ / C)

# Solve H Δx ≈ z directly; least squares avoids squaring the Jacobian condition number.
prefit_update, _, prefit_rank, prefit_singular_values = np.linalg.lstsq(
    prefit_jacobian,
    prefit_residuals_hz[:, 0],
    rcond=None,
)

observation_index = pd.Index(np.arange(1, observation_count + 1), name="Observation")
display(
    Markdown(
        r"At the initial state $\mathbf{x}_0$, the prefit residual is "
        r"$\mathbf{z}=\mathbf{y}-\mathbf{h}(\mathbf{x}_0)$ and "
        r"$\mathbf{z}\approx\mathbf{H}\Delta\mathbf{x}$. "
        "The Jacobian columns are position corrections in ECEF meters and receiver clock drift in m/s."
    )
)
print(f"Residual vector shape: {prefit_residuals_hz.shape}; Jacobian shape: {prefit_jacobian.shape}")

prefit_jacobian_columns = [
    "d fD/dx (Hz/m)",
    "d fD/dy (Hz/m)",
    "d fD/dz (Hz/m)",
    "d fD/db_u (Hz/(m/s))",
]


def preview_table(values, columns):
    if observation_count <= 10:
        return pd.DataFrame(values, index=observation_index, columns=columns)

    first_rows = pd.DataFrame(
        values[:5],
        index=pd.RangeIndex(1, 6, name="Observation"),
        columns=columns,
    )
    omitted_rows = pd.DataFrame(
        [["..."] * len(columns)],
        index=["..."],
        columns=columns,
    )
    last_rows = pd.DataFrame(
        values[-5:],
        index=pd.RangeIndex(observation_count - 4, observation_count + 1, name="Observation"),
        columns=columns,
    )
    return pd.concat([first_rows, omitted_rows, last_rows])


with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.precision", 4):
    display(preview_table(prefit_residuals_hz, ["Prefit residual (Hz)"]))
    display(preview_table(prefit_jacobian, prefit_jacobian_columns))

assert prefit_residuals_hz.shape == (observation_count, 1)
assert prefit_jacobian.shape == (observation_count, 4)
assert prefit_rank == 4, "The prefit Jacobian does not determine all four state corrections."
print(f"Least-squares method: numpy.linalg.lstsq; Jacobian rank: {prefit_rank} / 4")
print(f"One-step position correction (m): {prefit_update[:3]}")
print(f"One-step clock-drift correction (m/s): {prefit_update[3]:.6f}")
print(
    "Linearized residual RMS (Hz): "
    f"{np.linalg.norm(prefit_jacobian @ prefit_update - prefit_residuals_hz[:, 0]) / np.sqrt(observation_count):.4f}"
)

At the initial state $\mathbf{x}_0$, the prefit residual is $\mathbf{z}=\mathbf{y}-\mathbf{h}(\mathbf{x}_0)$ and $\mathbf{z}\approx\mathbf{H}\Delta\mathbf{x}$. The Jacobian columns are position corrections in ECEF meters and receiver clock drift in m/s.

Residual vector shape: (37, 1); Jacobian shape: (37, 4)


,Prefit residual (Hz)
1,12.6374
2,12.2664
3,11.9811
4,12.0282
5,12.142
...,...
33,41.8517
34,40.2442
35,38.825
36,37.0433


,d fD/dx (Hz/m),d fD/dy (Hz/m),d fD/dz (Hz/m),d fD/db_u (Hz/(m/s))
1,0.0091,0.0071,0.0057,-5.255
2,0.0096,0.008,0.0062,-5.255
3,0.0102,0.009,0.0068,-5.255
4,0.0107,0.0103,0.0074,-5.255
5,0.0113,0.0117,0.0083,-5.255
...,...,...,...,...
33,-0.0173,-0.0005,-0.0011,-5.255
34,-0.0158,-0.0008,-0.0014,-5.255
35,-0.0144,-0.001,-0.0017,-5.255
36,-0.0132,-0.0012,-0.0018,-5.255


Least-squares method: numpy.linalg.lstsq; Jacobian rank: 4 / 4
One-step position correction (m): [-1203.02335523   904.35967645  -714.32647825]
One-step clock-drift correction (m/s): -4.004977
Linearized residual RMS (Hz): 0.2117


## 3. Results

The map traces the initial receiver guess and each least-squares update; marker colors indicate iteration, and hovering shows position error relative to the simulated receiver. The residual plot shows observed-minus-modeled Doppler after the final position and clock update, with the injected noise standard deviation marked. This is a synthetic experiment: real-world orbit error, atmospheric propagation, receiver motion, satellite clock error, and multipath are not included.

In [8]:
# Replay the solver from its saved a priori state to retain every position for the convergence map.
map_estimate_position_ecef_m = initial_position_ecef_m.copy()
map_estimate_clock_bias_m_s = initial_clock_bias_m_s
map_iteration_positions_ecef_m = [map_estimate_position_ecef_m.copy()]

for _ in range(15):
    map_predicted_doppler_hz = doppler_model(
        map_estimate_position_ecef_m,
        map_estimate_clock_bias_m_s,
        satellite_position_km,
        satellite_velocity_km_s,
        observation_times,
    )
    map_residual_hz = observed_doppler_hz - map_predicted_doppler_hz
    map_jacobian = np.empty((map_residual_hz.size, 4))

    for coordinate in range(3):
        offset_m = np.zeros(3)
        offset_m[coordinate] = position_difference_step_m
        doppler_plus = doppler_model(
            map_estimate_position_ecef_m + offset_m,
            map_estimate_clock_bias_m_s,
            satellite_position_km,
            satellite_velocity_km_s,
            observation_times,
        )
        doppler_minus = doppler_model(
            map_estimate_position_ecef_m - offset_m,
            map_estimate_clock_bias_m_s,
            satellite_position_km,
            satellite_velocity_km_s,
            observation_times,
        )
        map_jacobian[:, coordinate] = (doppler_plus - doppler_minus) / (2.0 * position_difference_step_m)

    map_jacobian[:, 3] = -(CARRIER_HZ / C)
    map_update, _, map_rank, _ = np.linalg.lstsq(map_jacobian, map_residual_hz, rcond=None)
    map_estimate_position_ecef_m += map_update[:3]
    map_estimate_clock_bias_m_s += map_update[3]
    map_iteration_positions_ecef_m.append(map_estimate_position_ecef_m.copy())
    if np.linalg.norm(map_update[:3]) < 1e-3 and abs(map_update[3]) < 1e-6:
        break

map_final_doppler_hz = doppler_model(
    map_estimate_position_ecef_m,
    map_estimate_clock_bias_m_s,
    satellite_position_km,
    satellite_velocity_km_s,
    observation_times,
)
map_postfit_residual_hz = observed_doppler_hz - map_final_doppler_hz
np.testing.assert_allclose(map_estimate_position_ecef_m, estimate_position_ecef_m, atol=1e-6)

map_geodetic_positions = np.array(
    [ecef_to_geodetic(position) for position in map_iteration_positions_ecef_m]
)
map_latitudes_deg = map_geodetic_positions[:, 0]
map_longitudes_deg = map_geodetic_positions[:, 1]
map_position_errors_m = np.linalg.norm(
    np.asarray(map_iteration_positions_ecef_m) - true_position_ecef_m,
    axis=1,
)
map_step_labels = ["Initial guess"] + [f"Iteration {index}" for index in range(1, len(map_latitudes_deg))]
map_custom_data = np.column_stack((map_step_labels, map_position_errors_m))
true_latitude_deg, true_longitude_deg, _ = ecef_to_geodetic(true_position_ecef_m)

convergence_map = go.Figure()
convergence_map.add_trace(
    go.Scattergeo(
        lon=map_longitudes_deg,
        lat=map_latitudes_deg,
        mode="lines+markers",
        name="Position estimates",
        line={"color": "#66777d", "width": 2},
        marker={
            "size": 11,
            "color": np.arange(len(map_step_labels)),
            "colorscale": "Viridis",
            "cmin": 0,
            "cmax": max(1, len(map_step_labels) - 1),
            "colorbar": {
                "title": "Iteration",
                "tickmode": "array",
                "tickvals": list(range(len(map_step_labels))),
                "ticktext": map_step_labels,
            },
        },
        customdata=map_custom_data,
        hovertemplate=(
            "%{customdata[0]}<br>Latitude: %{lat:.6f}°<br>Longitude: %{lon:.6f}°"
            "<br>Error from truth: %{customdata[1]:.1f} m<extra></extra>"
        ),
    )
)
convergence_map.add_trace(
    go.Scattergeo(
        lon=[true_longitude_deg],
        lat=[true_latitude_deg],
        mode="markers+text",
        name="Simulated receiver",
        text=["Truth"],
        textposition="bottom right",
        marker={"size": 13, "color": "#bd3b32", "symbol": "star", "line": {"color": "white", "width": 1}},
        hovertemplate="Simulated receiver<br>Latitude: %{lat:.6f}°<br>Longitude: %{lon:.6f}°<extra></extra>",
    )
)
convergence_map.update_geos(
    fitbounds="locations",
    projection_type="mercator",
    showland=True,
    landcolor="#e7e5dc",
    showocean=True,
    oceancolor="#dcecf0",
    showcoastlines=True,
    coastlinecolor="#68777a",
    showcountries=True,
    countrycolor="#a4aaa5",
    lataxis_showgrid=True,
    lonaxis_showgrid=True,
)
convergence_map.update_layout(
    title="Receiver-position convergence across least-squares iterations",
    height=600,
    margin={"l": 10, "r": 10, "t": 55, "b": 10},
    legend={"orientation": "h", "yanchor": "bottom", "y": 0.01, "xanchor": "left", "x": 0.01},
)
convergence_map.show()

elapsed_minutes = sample_seconds / 60.0
residual_figure = go.Figure()
residual_figure.add_trace(
    go.Scatter(
        x=elapsed_minutes,
        y=map_postfit_residual_hz,
        mode="lines+markers",
        name="Observed − modeled",
        line={"color": "#137c8b", "width": 1.5},
        marker={"size": 6},
    )
)
residual_figure.add_hrect(
    y0=-NOISE_STD_HZ,
    y1=NOISE_STD_HZ,
    fillcolor="#e07a32",
    opacity=0.14,
    line_width=0,
    annotation_text="Injected noise ±1σ",
    annotation_position="top left",
)
residual_figure.add_hline(y=0.0, line={"color": "#555555", "width": 1})
residual_figure.update_layout(
    title="Post-fit Doppler residuals",
    xaxis_title="Time from start of visible pass (min)",
    yaxis_title="Observed − modeled (Hz)",
    height=430,
    template="plotly_white",
    hovermode="x unified",
    margin={"l": 70, "r": 25, "t": 65, "b": 55},
)
residual_figure.show()

print(f"Mapped positions: {len(map_iteration_positions_ecef_m)} (initial guess plus updates)")
print("Position error by step (m): " + ", ".join(f"{error:.1f}" for error in map_position_errors_m))
print(f"Post-fit residual RMS: {np.sqrt(np.mean(map_postfit_residual_hz**2)):.3f} Hz")

Mapped positions: 4 (initial guess plus updates)
Position error by step (m): 1655.3, 15.3, 19.3, 19.3
Post-fit residual RMS: 0.208 Hz
